# ⚡ Gate-Kernel Fusion Pilot — Triton token gate fused into Teo v2's forward pass

Spec: `docs/GATE_KERNEL_FUSION_SPEC.md`. Fuses `action_boundary.sl`'s token
gate directly into the `lm_head` matmul of `NetelproTransformer` (Teo v2)
via a hand-written Triton kernel, instead of the two-kernel
matmul-then-mask path that's the only thing that exists today.

**What this notebook measures, honestly:** baseline (unfused: matmul + separate
mask, today's actual `filter_logits_tensor` path) vs. fused (one Triton
kernel) vs. ungated (matmul alone, the floor). Real `torch.cuda.Event`
timings, both a broad-allowed-range config and a narrow-allowed-range config
(tile-skip matters more when most of the vocab is masked — open question 1
in the spec). No numbers claimed before they're measured here.

## 1. Hardware check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
assert torch.cuda.is_available(), "Este pilot necesita GPU real -- activa el acelerador GPU T4/P100 en Kaggle."

try:
    import triton
    print(f"Triton: {triton.__version__}")
except ImportError as e:
    raise RuntimeError(
        "Triton no está instalado en esta imagen de Kaggle. "
        "Intenta `!pip install -q triton` en la celda siguiente si esto falla."
    ) from e


## 2. Clonar/actualizar el repo e instalar

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('netelpro'):
    !git clone https://github.com/jona2428/netelpro.git
else:
    !cd /kaggle/working/netelpro && git pull origin master
%cd /kaggle/working/netelpro
!pip install -q -e .


## 3. Construir Teo v2 en GPU

Config real de `docs/ROADMAP_TEO_7B_MOE_SDS.md` (vocab 32768, 12 capas, 12
cabezas, d_model 768). Init aleatorio -- este pilot mide latencia del kernel,
no calidad del modelo, así que no hace falta descargar ningún checkpoint.

In [ ]:
import torch
from netelpro.neuro.transformer import NetelproTransformer, NetelproTransformerConfig
from netelpro.neuro.gate_kernel import HAS_TRITON, gated_lm_head, gated_lm_head_reference
from netelpro.neuro.native_kernel import NetelproVectorKernel
from pathlib import Path

assert HAS_TRITON, "HAS_TRITON es False -- revisa la instalación de triton en esta imagen."

config = NetelproTransformerConfig(
    vocab_size=32768, block_size=512, n_layer=12, n_head=12, n_embd=768,
)
model = NetelproTransformer(config).cuda().eval()
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parámetros")

vk = NetelproVectorKernel(Path("netelpro/neuro/rules/action_boundary.sl"))


## 4. Chequeo de correctitud en GPU real (antes de confiar en cualquier número)

Compara el kernel Triton fusionado contra la referencia torch (misma regla,
dos implementaciones) sobre tensores CUDA reales -- el test de CI
(`tests/test_gate_kernel.py`) ya corrió esto en CPU con la referencia sola;
esto es lo que falta: el kernel Triton de verdad, lanzado en GPU.

In [ ]:
torch.manual_seed(0)
x = torch.randn(config.n_embd, device="cuda")
weight = model.lm_head.weight.detach()

test_ranges = [(0, config.vocab_size - 1, 1), (100, 30000, 1), (5000, 5200, 1), (0, config.vocab_size - 1, 0)]

all_ok = True
for lo, hi, safety in test_ranges:
    fused = gated_lm_head(x, weight, lo, hi, safety)
    ref = gated_lm_head_reference(x, weight, lo, hi, safety)
    ok = torch.allclose(fused, ref, atol=1e-3, rtol=1e-3, equal_nan=False) if safety == 1 else torch.equal(fused, ref)
    all_ok &= ok
    print(f"range=({lo},{hi}) safety={safety}: match={ok}, max_abs_diff={(fused-ref).abs().max().item():.6f}")

assert all_ok, "El kernel Triton no coincide con la referencia -- NO confíes en los tiempos de la sección 5 hasta arreglar esto."
print("\nCorrectitud confirmada en GPU real. Procediendo al benchmark.")


## 5. Benchmark de tres vías (números reales, no prometidos)

Para cada config de rango: `ungated` (matmul solo, el piso), `baseline`
(matmul + `filter_logits_tensor`, el path real de hoy), `fused` (kernel
Triton, un solo lanzamiento). `torch.cuda.Event` para timing real de GPU,
promedio sobre varios pasos con warmup.

In [ ]:
import time

def bench_step(fn, n_warmup=20, n_iters=200):
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    times_ms = []
    for _ in range(n_iters):
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))
    times_ms.sort()
    return {
        "mean_ms": sum(times_ms) / len(times_ms),
        "min_ms": times_ms[0],
        "max_ms": times_ms[-1],
        "p50_ms": times_ms[len(times_ms) // 2],
    }

x_last = torch.randn(config.n_embd, device="cuda")
weight = model.lm_head.weight.detach()

range_configs = {
    "broad (allowed [100, 30000] of 32768 -- mostly allowed)": (100, 30000, 1),
    "narrow (allowed [5000, 5200] of 32768 -- mostly masked)": (5000, 5200, 1),
}

results = {}
for label, (lo, hi, safety) in range_configs.items():
    results[label] = {
        "ungated": bench_step(lambda: torch.nn.functional.linear(x_last, weight)),
        "baseline_unfused": bench_step(lambda: vk.filter_logits_tensor(
            torch.nn.functional.linear(x_last, weight).unsqueeze(0), lo, hi, safety
        )),
        "fused_triton": bench_step(lambda: gated_lm_head(x_last, weight, lo, hi, safety)),
    }

import json
print(json.dumps(results, indent=2))


## 6. Tabla de resultados

In [ ]:
import pandas as pd

rows = []
for range_label, methods in results.items():
    for method, m in methods.items():
        rows.append({"range": range_label, "method": method, **m})

df = pd.DataFrame(rows)
df


## 7. Lectura honesta

Completar después de correr:
- ¿El kernel fusionado (`fused_triton`) fue más rápido que `baseline_unfused`? ¿Por cuánto, en cada config de rango?
- ¿La config `narrow` (más tiles para saltar) mostró más ganancia que `broad`, como predice la pregunta abierta 1 del spec?
- ¿Qué tan cerca quedó `fused_triton` de `ungated` (el piso teórico sin overhead de gate)?

Pegar la tabla real de la celda 6 en `docs/GATE_KERNEL_FUSION_SPEC.md` Sección 8 cuando esté corrido -- no antes.